# Ways to improve Kokoro

- Text Processing
- Sentence-Level Generation
- Peak Normalization
- Fine Tuning

# Installing and import Kokoro

In [7]:
!pip install -q "kokoro>=0.9.4" soundfile
!apt-get -qq update
!apt-get -qq install -y espeak-ng


[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
import re
import torch
import soundfile as sf

from kokoro import KPipeline
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"

# American English
pipeline = KPipeline(lang_code="a")

print("Kokoro loaded. Device:", device)

Kokoro loaded. Device: cpu


# Running the model

In [3]:
text = "Hello! This is Kokoro text to speech. The model is now running."

generator = pipeline(
    text,
    voice="af_heart"
)

for i, (gs, ps, audio) in enumerate(generator):
    display(Audio(audio, rate=24000))
    sf.write(f"kokoro_output_{i}.wav", audio, 24000)

c:\Users\mazin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mazin\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


voices/af_heart.pt: reconstructing file:   0%|          |  0.00B /  523kB            

voices/af_heart.pt: downloading bytes:           |  0.00B            

c:\Users\mazin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mazin\.cache\huggingface\hub\models--hexgrad--Kokoro-82M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


# Improving Text Processing

In this we basically, clean whitespace, expands commond abbreviations and converts numbers into spoken words before synthesis.

In [4]:
!pip install -q num2words


[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from num2words import num2words

def preprocess_text(text):
    text = re.sub(r"\s+", " ", text).strip()

    abbreviations = {
        r"\bDr\.": "Doctor",
        r"\bMr\.": "Mister",
        r"\bMrs\.": "Misses",
        r"\bSt\.": "Street",
    }

    for pattern, replacement in abbreviations.items():
        text = re.sub(pattern, replacement, text)

    # Convert standalone integers to words
    text = re.sub(
        r"\b\d+\b",
        lambda m: num2words(int(m.group())),
        text
    )

    return text

text = preprocess_text(
    "Dr. Smith has 25 books. He lives on St. James Street."
)

print(text)

Doctor Smith has twenty-five books. He lives on Street James Street.


# Improve Inference and Chunking

We will be generating sentence by sentence, experimenting with speed and normalizing the output audio

In [6]:
def generate_speech(text, voice="af_heart", speed=1.0):
    text = preprocess_text(text)

    sentences = re.split(r"(?<=[.!?])\s+", text)
    audio_chunks = []

    for sentence in sentences:
        if not sentence.strip():
            continue

        generator = pipeline(
            sentence,
            voice=voice,
            speed=speed
        )

        for _, _, audio in generator:
            audio_chunks.append(audio)

    if not audio_chunks:
        raise ValueError("No speech was generated.")

    audio = torch.cat([
        torch.as_tensor(chunk).flatten()
        for chunk in audio_chunks
    ]).numpy()

    # Gentle peak normalization
    peak = abs(audio).max()
    if peak > 0:
        audio = audio * (0.95 / peak)

    sf.write("improved_kokoro.wav", audio, 24000)
    display(Audio(audio, rate=24000))

    return audio

audio = generate_speech(
    "Hello! Today we will test the quality of Kokoro. It can read 25 books.",
    voice="af_heart",
    speed=1.0
)

# Fine Tuning

For fine-tuning, we use clean recordings paired with accurate transcripts. We can keep each recording short. 
#### Since I don't have the dataset now, So can't do fine tuning now

# We can also consider using Pronunciation correction using a custom lexicon. 
#### Our existing preprocessing doesn't handle names, acronyms, technical terms etc. We can consider creating a pronunication dictionary for words that Kokoro repeatedly mispronounces, and convert only those words before synthesis. 